In [43]:
# Modules
import numpy as np
import pandas as pd
import itertools
import destruction_models as models
import tensorflow as tf
import random

from destruction_utilities import *
from destruction_statistics import *
from numpy import random
import matplotlib.pyplot as plt
from keras import callbacks, preprocessing
from keras.utils.all_utils import Sequence
from keras.metrics import CategoricalAccuracy, Precision
from keras.models import load_model
from sklearn.metrics import precision_recall_curve, roc_auc_score
from os import path
import zarr

In [44]:
CITY = 'aleppo'

In [45]:
def get_zarr(city, type, set):
    path = f'../data/{city}/others/{city}_{type}s_{set}.zarr'
    return zarr.open(path, mode='r')

In [4]:
# train_path = f'../data/{CITY}/others/{CITY}_images_train.zarr'
# validate_path = f'../data/{CITY}/others/{CITY}_images_validate.zarr'
# test_path = f'../data/{CITY}/others/{CITY}_images_test.zarr'

In [65]:
class ZarrGenerator(Sequence):
    def __init__(self, images, labels, batch_size=32):
        self.images = images
        self.labels = labels
        self.batch_size = batch_size
        
    def __len__(self):
        return len(self.images)//self.batch_size
    
    def __getitem__(self, index):
        
#         n = self.images.shape[0]
#         indices = np.random.choice(np.arange(n),32)
        
#         X = self.images[indices]
#         y = self.labels[indices]
        X = self.images[index*self.batch_size:(index+1)*self.batch_size]
        y = self.labels[index*self.batch_size:(index+1)*self.batch_size]
        
        return self.augment(X), y.flatten()
    
    def augment(self, X):
        # Horizontal and vertical flip
        flipping_funcs = [
            lambda image: image,
            lambda image: np.fliplr(image),
            lambda image: np.flipud(image),
            lambda image: np.flipud(np.fliplr(image))
        ]
        func = random.choice(flipping_funcs)
        X = func(X)
        
        # Brightness
        alpha = random.choice(np.linspace(0.85, 1.4))
        X = X * alpha
        
        return X

In [60]:
training_images = get_zarr(CITY, 'image', 'train')
training_labels = get_zarr(CITY, 'label', 'train')
validation_images = get_zarr(CITY, 'image', 'validate')
validation_labels = get_zarr(CITY, 'label', 'validate')
test_images = get_zarr(CITY, 'image', 'test')
test_labels = get_zarr(CITY, 'label', 'test')

In [61]:
print(training_images.shape, training_labels.shape, np.unique(training_labels, return_counts=True))
print(validation_images.shape, validation_labels.shape, np.unique(validation_labels, return_counts=True))
print(test_images.shape, test_labels.shape, np.unique(test_labels, return_counts=True))


(320514, 128, 128, 3) (320514, 1, 1, 1) (array([0., 1.]), array([303648,  16866]))
(68692, 128, 128, 3) (68692, 1, 1, 1) (array([0., 1.]), array([65059,  3633]))
(68366, 128, 128, 3) (68366, 1, 1, 1) (array([0., 1.]), array([65288,  3078]))


In [11]:
16866/303648

0.055544577932342715

In [66]:
training_generator = ZarrGenerator(training_images, training_labels, batch_size=32)
validation_generator = ZarrGenerator(validation_images, validation_labels, batch_size=32)
test_generator = ZarrGenerator(test_images, test_labels, batch_size=32)

In [13]:
# https://datascience.stackexchange.com/questions/45165/how-to-get-accuracy-f1-precision-and-recall-for-a-keras-model
# Callbacks
training_callbacks = [
    callbacks.EarlyStopping(monitor='val_accuracy', restore_best_weights=True, patience=5),
    callbacks.BackupAndRestore(backup_dir='../models')
]

args  = dict(shape=(128, 128, 3), filters=16, units=32, dropout=0.1) # ! Check parameters before run
model = models.convolutional_network(**args)
model.compile(optimizer='adam', loss='binary_focal_crossentropy', metrics=['accuracy'])


# Train model on dataset
model.fit_generator(
    training_generator,
    validation_data=validation_generator, 
    epochs=100, 
    callbacks = training_callbacks
)


# model.save(f'../models/{CITY}_unbalanced', save_format="h5")
# train_generator.__getitem__(1)[0].shape

/var/folders/z1/7pydgng971nfzdf2rwg_p__r0000gn/T/ipykernel_26857/765512165.py:17: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  model.fit_generator(


Epoch 1/100


2022-06-13 16:07:16.944088: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


10016/10016 [==============================] - ETA: 0s - loss: 0.0579 - accuracy: 0.9493

2022-06-13 18:45:51.992658: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


10016/10016 [==============================] - 9965s 995ms/step - loss: 0.0579 - accuracy: 0.9493 - val_loss: 0.0550 - val_accuracy: 0.9519
Epoch 2/100
10016/10016 [==============================] - 9938s 992ms/step - loss: 0.0549 - accuracy: 0.9523 - val_loss: 0.0550 - val_accuracy: 0.9519
Epoch 3/100
10016/10016 [==============================] - 10122s 1s/step - loss: 0.0547 - accuracy: 0.9523 - val_loss: 0.0546 - val_accuracy: 0.9519
Epoch 4/100
10016/10016 [==============================] - 10447s 1s/step - loss: 0.0546 - accuracy: 0.9523 - val_loss: 0.0547 - val_accuracy: 0.9519
Epoch 5/100
10016/10016 [==============================] - 11088s 1s/step - loss: 0.0545 - accuracy: 0.9523 - val_loss: 0.0546 - val_accuracy: 0.9519
Epoch 6/100
10016/10016 [==============================] - 10573s 1s/step - loss: 0.0545 - accuracy: 0.9523 - val_loss: 0.0547 - val_accuracy: 0.9519


In [49]:
model = load_model(f'../models/{CITY}_unbalanced')

In [50]:
import gc
gc.collect()

batch_size = 5000
iters = test_images.shape[0] // batch_size
preds = []
labels = []
for i in range(0, iters):
    end = (i+1)*batch_size
    print(i)
    print(i*batch_size, end, "\n")
    if i == iters - 1:
        preds.append(model.predict(test_images[i*batch_size:]))
        labels.append(test_labels[i*batch_size:])
    else:
        preds.append(model.predict(test_images[i*batch_size: end]))
        labels.append(test_labels[i*batch_size: end])

0
0 5000 



2022-06-14 12:54:25.105073: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


1
5000 10000 

2
10000 15000 

3
15000 20000 

4
20000 25000 

5
25000 30000 

6
30000 35000 

7
35000 40000 

8
40000 45000 

9
45000 50000 

10
50000 55000 

11
55000 60000 

12
60000 65000 



In [51]:
yhat = np.squeeze(np.concatenate(preds, axis=0))
y = np.squeeze(np.concatenate(labels, axis=0 ))

In [52]:
yhat.shape

(68366,)

In [53]:
roc_auc_score(y, yhat)

0.5752507990984554

In [55]:
a = get_zarr(CITY, 'image', 'train')

In [56]:
a.shape

(320514, 128, 128, 3)

In [67]:
import time

start = time.time()

training_generator.__getitem__(0)

(array([[[[ 25.73979592,  12.35510204,   8.23673469],
          [ 25.73979592,  12.35510204,   8.23673469],
          [ 33.97653061,  20.59183673,  16.47346939],
          ...,
          [ 76.18979592,  54.56836735,  50.45      ],
          [ 76.18979592,  54.56836735,  50.45      ],
          [ 76.18979592,  58.68673469,  50.45      ]],
 
         [[ 42.21326531,  24.71020408,  16.47346939],
          [ 33.97653061,  20.59183673,  16.47346939],
          [ 33.97653061,  24.71020408,  16.47346939],
          ...,
          [ 76.18979592,  54.56836735,  50.45      ],
          [ 76.18979592,  54.56836735,  50.45      ],
          [ 76.18979592,  54.56836735,  50.45      ]],
 
         [[ 50.45      ,  32.94693878,  25.73979592],
          [ 42.21326531,  24.71020408,  25.73979592],
          [ 42.21326531,  28.82857143,  25.73979592],
          ...,
          [ 76.18979592,  54.56836735,  50.45      ],
          [ 67.95306122,  50.45      ,  50.45      ],
          [ 67.95306122,  46.33

In [70]:
n = training_images.shape[0]
indices = np.random.choice(np.arange(n),32)
        
# X = self.images[indices]
# y = self.labels[indices]

indices

array([197394, 320305, 234631,  17968,  25798, 309186, 245652, 291177,
       253509, 118715,  16095, 272819,  86557, 299533,  87978, 272951,
       264912, 167461, 318867, 253662,  22594,  54764, 226074, 109139,
        90076, 116489, 170743, 231972, 138009,  49008,  32374,  34115])

In [69]:
training_images[indices]

IndexError: unsupported selection item for basic indexing; expected integer or slice, got <class 'numpy.ndarray'>